# QLoRA recipe for `json_extract`

Fine-tune a small instruct model so it emits the JSON object evalkit scores, then measure it with the same YAML task. This notebook is a recipe for a Colab T4. It does not contain a training curve, a loss number, or an accuracy.

Train on the rows in this notebook. They are format examples, not the tickets in `tasks/json_extract.yaml`. Scoring a model on tickets it was trained on is not a result worth keeping.

After training, serve the base model and the adapter on any OpenAI-compatible server and run:

```bash
evalkit run tasks/json_extract.yaml --base-url http://127.0.0.1:8000/v1 --model your-model --api-key local
```

Keep the leaderboard evalkit prints. Do not copy a number from someone else's run into a write-up of this notebook.


## 1. Check the GPU

A T4 is enough for Llama 3.2 1B in 4-bit. If this cell does not see CUDA, switch the Colab runtime to GPU before continuing. A CPU run of this notebook is not the recipe.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. In Colab: Runtime → Change runtime type → T4 GPU.")
print(torch.cuda.get_device_name(0))


## 2. Install Unsloth and TRL

Unsloth's Colab installer changes. If the import in the next cell fails, use the command currently published at <https://unsloth.ai/docs/get-started/install/google-colab> and rerun from here.

This cell installs the libraries. It does not pin a score, and it does not download a result file.


In [ ]:
import sys

!{sys.executable} -m pip install --upgrade "unsloth" "trl" "peft" "accelerate" "bitsandbytes" "datasets" "transformers"


## 3. Load a 4-bit base and attach LoRA

QLoRA here means: the base weights stay 4-bit, and the trainer updates a low-rank adapter. `r` and `lora_alpha` are a small default for a smoke run, not a tuned configuration. Swap the model name for `unsloth/Llama-3.2-3B-Instruct` if the T4 has room and you want a larger base.


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 1024
model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


## 4. Build a tiny supervised set

Same contract as the eval task: one JSON object with `name`, `email`, `order_id`, and `issue`. These tickets are new. Leave the eval file out of this list.


In [ ]:
import json

from datasets import Dataset

SYSTEM = (
    "You extract structured fields from a support ticket. "
    "Reply with a single JSON object and no other text. "
    "Keys: name (string), email (string), order_id (string), issue (string)."
)

ROWS = [
    (
        "Mina Patel (mina.patel@example.com) says order #K-15 arrived with a cracked screen.",
        {"name": "Mina Patel", "email": "mina.patel@example.com", "order_id": "K-15", "issue": "cracked screen"},
    ),
    (
        "Owen Blake (owen.blake@example.com) was sent the wrong color for order #L-204.",
        {"name": "Owen Blake", "email": "owen.blake@example.com", "order_id": "L-204", "issue": "wrong color"},
    ),
    (
        "Hana Ito (hana.ito@example.com) never got a tracking number for order #M-8.",
        {"name": "Hana Ito", "email": "hana.ito@example.com", "order_id": "M-8", "issue": "missing tracking number"},
    ),
    (
        "Elena Costa (elena.costa@example.com) wants order #N-77 marked as a gift.",
        {"name": "Elena Costa", "email": "elena.costa@example.com", "order_id": "N-77", "issue": "mark as gift"},
    ),
    (
        "Samir Haddad (samir.haddad@example.com) received an empty box for order #P-310.",
        {"name": "Samir Haddad", "email": "samir.haddad@example.com", "order_id": "P-310", "issue": "empty box"},
    ),
    (
        "Ruth Adler (ruth.adler@example.com) needs a VAT invoice for order #Q-2.",
        {"name": "Ruth Adler", "email": "ruth.adler@example.com", "order_id": "Q-2", "issue": "VAT invoice"},
    ),
]


def as_text(ticket: str, answer: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Ticket:\n" + ticket},
        {"role": "assistant", "content": json.dumps(answer)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


dataset = Dataset.from_list([{"text": as_text(ticket, answer)} for ticket, answer in ROWS])
print(dataset[0]["text"][:400])


## 5. Train

`max_steps=40` is only there so the cell finishes on a free GPU. It is not a recommended training length. `report_to="none"` keeps the run local.

TRL has used both `tokenizer` and `processing_class`, and `SFTConfig` has used both `max_seq_length` and `max_length`. The fallbacks below follow the signature that imports, so a Colab pin drift fails in the constructor instead of halfway through a silent mismatch.


In [ ]:
from trl import SFTConfig, SFTTrainer

config_kwargs = dict(
    output_dir="outputs",
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=40,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)

try:
    args = SFTConfig(max_seq_length=max_seq_length, **config_kwargs)
except TypeError:
    args = SFTConfig(max_length=max_seq_length, **config_kwargs)

trainer_kwargs = {"model": model, "train_dataset": dataset, "args": args}
try:
    trainer = SFTTrainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = SFTTrainer(**trainer_kwargs, tokenizer=tokenizer)

trainer.train()


## 6. Read one completion

This cell prints text. It does not score it. Use evalkit for that, on tickets that were not in `ROWS`.


In [ ]:
FastLanguageModel.for_inference(model)

probe = "Casey Nguyen (casey.nguyen@example.com) says order #R-19 is missing a charger."
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Ticket:\n" + probe},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True))


## 7. Save the adapter

This writes the LoRA adapter and tokenizer next to the notebook. Serving it is a separate step: vLLM, Ollama, or another OpenAI-compatible server. Point `evalkit run` at that server. Compare the base model and the adapter as two targets in one task file if you want them on the same leaderboard.

The Next.js dashboard shipped with evalkit reads a committed fixture file. It will not show this adapter until you generate your own results and choose to publish them, labeled as what they are.


In [ ]:
model.save_pretrained("json-extract-lora")
tokenizer.save_pretrained("json-extract-lora")
print("saved json-extract-lora")
